# 08 — Complete Autonomy Contract and Controlled Hill Climbing

**Chapter 2 teaching lab — §§2.8–2.14, centered on §2.11**

> **Teaching outcome:** Compose an autonomy contract and practice controlled hill climbing.

Flow: *concept → why it matters → diagram → read the implementation → expected behavior →
red-team round → playground → challenge.*

In [ ]:
# Setup: make harnesslab/ importable from any working directory, then load this lab's controls.
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "harnesslab").is_dir())
sys.path.insert(0, str(ROOT))

from harnesslab import Action, AutonomyLevel, Decision, EXECUTES, Reversibility
import copy, json
from harnesslab.contract import (Scenario, bounded_notification_grant, decide, diff_decisions,
                                 load_contract, validate_contract)
from harnesslab.slo import DEMO_PROPOSALS, metrics, run_controlled, slo_report
from harnesslab.viz import card, checks, decisions, diff_view, pill, slo_cards
from harnesslab.playground import contract_playground
from harnesslab.arena import NAIVE, SECURE, run_gauntlet
from harnesslab.viz import gauntlet_board, journey, show_source
from IPython.display import display

journey(8)

## Concept
The previous notebooks each built one control. An **autonomy contract** composes them into one
machine-readable document per mission: the envelope ceiling, the action catalog with reversibility and
costs, approval rules, budgets, checkpoints, and stop conditions.

The contract is useful only if:
1. it is **validated** against structural invariants before use;
2. a single `decide()` function enforces **all** of it, not only the fields someone remembered;
3. every threat in the threat model maps to a **test**; and
4. changes are made by **controlled hill climbing**: one narrow change at a time, with a decision diff,
   evidence, and the previous version kept as the rollback target.

## Why this matters
Contracts drift. A field that nothing enforces (say, `max_level`) reads as a control but is not one.
Controlled hill climbing is how autonomy expands safely: small, measured, reversible policy changes.

## Diagram
```mermaid
flowchart TD
  C[contract.json] --> V{validate_contract}
  V -->|problems| X[refuse to run]
  V -->|valid| D
  subgraph D ["decide(): first match wins"]
    direction TB
    a1["unknown action → DENY"] --> a2["prohibited → DENY"] --> a3["external disabled → DENY"]
    a3 --> a4["over hard limit → DENY"] --> a5["irreversible → APPROVAL"] --> a6["over action limits → APPROVAL"]
    a6 --> a7["approval required → APPROVAL"] --> a8["above max_level → DENY unless bounded grant"]
    a8 --> a9["otherwise ALLOW, or MONITOR past the warning"]
  end
```

In [ ]:
contract = load_contract()
assert validate_contract(contract) == [], validate_contract(contract)
print(json.dumps(contract["autonomy"]), "| budget", contract["budget"])
show_source(decide)

## Decision table

In [ ]:
decisions([(name, *decide(name, contract)) for name in contract["actions"]], title="research_agent v1")

## Expected behavior: threat-model-to-test checks (§2.12)
Each row names a threat and the assertion that proves the contract handles it.

In [ ]:
THREATS = {
    "agent deletes records":            decide("delete_records", contract)[0] is Decision.DENY,
    "agent publishes without review":   decide("publish_report", contract)[0] is Decision.REQUIRE_APPROVAL,
    "agent invents a new tool":         decide("wire_transfer", contract)[0] is Decision.DENY,
    "run overspends its budget":        decide("stage_report", contract, spent=29)[0] is Decision.DENY,
    "run nears its budget unnoticed":   decide("search", contract, spent=20)[0] is Decision.MONITOR,
    "stale approvals keep working":     "approval_expired" in contract["stop_conditions"],
    "world changes mid-run":            "context_drift" in contract["stop_conditions"],
    "irreversible step, no checkpoint": contract["checkpoints"]["before_irreversible_action"],
}
assert all(THREATS.values())
checks([(t, ok, "") for t, ok in THREATS.items()], title="Threat model", good="covered", bad="GAP")

## Contracts that look fine but are not
`validate_contract` must reject edits that silently remove a control.

In [ ]:
WEAKENINGS = {
    "publish without approval": lambda c: c["actions"]["publish_report"].pop("approval"),
    "drift stop removed":       lambda c: c["stop_conditions"].remove("context_drift"),
    "warning above hard limit": lambda c: c["budget"].update(warning=40),
    "unbounded grant":          lambda c: c["actions"]["internal_notification"].update(approval="not_required"),
}
rows = []
for label, weaken in WEAKENINGS.items():
    c = copy.deepcopy(contract)
    weaken(c)
    problems = validate_contract(c)
    assert problems, label
    rows.append((label, False, "; ".join(problems)))
checks(rows, title="Weakened contracts", bad="rejected")

## Controlled hill climbing (§2.11)
**Hypothesis:** internal notifications to small groups are low-risk, and the approval step is slowing the
team down. **Change:** grant `internal_notification` without approval, *bounded* to 10 recipients.
Larger sends still require approval. Nothing else changes.

```mermaid
flowchart LR
  v1[v1 contract] -->|one narrow change| v2[v2 contract]
  v2 --> DF[decision diff] & SL[SLOs under v2]
  DF & SL --> RB{review board}
  RB -->|evidence holds| PR[promote v2]
  RB -->|regression| RO[roll back to v1]
```

In [ ]:
v1 = contract
v2 = bounded_notification_grant(v1)
assert validate_contract(v2) == []

SCENARIOS = [Scenario(n) for n in v1["actions"]] + [
    Scenario("internal_notification", {"recipients": 3}),
    Scenario("internal_notification", {"recipients": 10}),
    Scenario("internal_notification", {"recipients": 250}),
    Scenario("internal_notification", {"recipients": 3}, spent=28),
    Scenario("publish_report", {"recipients": 3}),
]
changes = diff_decisions(v1, v2, SCENARIOS)
display(show_source(bounded_notification_grant))
diff_view(changes)

### Guardrails on the change

In [ ]:
# The change touched only the targeted action...
assert {s.action for s, _, _ in changes} == {"internal_notification"}
# ...never relaxed a denial or an irreversible action...
assert all(d1 is not Decision.DENY for _, d1, _ in changes)
assert all(v1["actions"][s.action]["reversibility"] != "irreversible" for s, _, _ in changes)
# ...and the bound holds: large sends still need approval, and the budget still applies.
assert decide("internal_notification", v2, args={"recipients": 250})[0] is Decision.REQUIRE_APPROVAL
assert decide("internal_notification", v2, args={"recipients": 3}, spent=28)[0] is Decision.DENY
ROLLBACK_TARGET = v1
print(f"{len(changes)} scenario(s) changed, all within the hypothesis. Rollback target: v1.")

### Evidence for the review board: SLOs under v1 and v2

In [ ]:
evidence = {}
for label, c in {"v1": v1, "v2": v2}.items():
    events, spent = run_controlled(DEMO_PROPOSALS, c, decide)
    evidence[label] = slo_report(metrics(events, spent, c))
    display(card(f"SLOs under {label}", slo_cards(evidence[label]).markup, accent="#0891b2"))

regressed = [k for k in evidence["v1"] if evidence["v1"][k][2] and not evidence["v2"][k][2]]
verdict = "rollback" if regressed else "promote" if all(ok for *_, ok in evidence["v2"].values()) else "hold"
assert verdict == "rollback" and regressed == ["high_risk_pause_rate"]
contract = ROLLBACK_TARGET
card("Review board verdict",
     f"{pill('deny', verdict.upper())} &nbsp; regressed: <code>{', '.join(regressed)}</code><br>"
     f"The diff was narrow and the bound held, but the evidence says v2 lets an action the SLOs classify "
     f"as <b>high-risk</b> run unreviewed. The live contract reverts to v1.", accent="#dc2626")

### Read the verdict carefully
Every guardrail on the *diff* passed, yet the *evidence* failed. `internal_notification` sits above the
envelope ceiling, so the SLO counts it as high-risk, and v2 now executes it without a pause. The
board is not saying the hypothesis is wrong. It is saying the proposer changed **authority** without
changing the **definition of risk** that the organization measures against. Both have to move
together, each as its own reviewed step.

This is controlled hill climbing working as intended: a plausible change was proposed, measured,
and rolled back automatically, before it reached production.

## The final gauntlet
Every attack from every notebook, against everything this chapter built.

In [ ]:
secure, shortcuts = run_gauntlet(SECURE), run_gauntlet(NAIVE)
assert all(r.blocked for r in secure)
gauntlet_board(secure, shortcuts, title=f"Chapter 2 Gauntlet — {len(secure)} attacks")

## Playground
Compare v1 and v2 for any action, budget position, and recipient count.

In [ ]:
contract_playground(v1, v2)

## Anti-patterns to recognize (§2.13)
| Anti-pattern | Where this lab counters it |
|---|---|
| Global `autonomous=True` switch | 02: multi-axis envelope |
| Denylist of dangerous tools | 01: default-deny allowlist |
| Agent-supplied risk labels | 03: harness-owned catalog |
| Approval of an intent, not a payload | 04: digest-bound, expiring approvals |
| Per-call limits with no run total | 05: capability budgets |
| Hand-entered control metrics | 06: SLOs computed from the event log |
| "It was approved when we started" | 07: leases, sealed checkpoints, drift review |
| Contract fields nothing enforces | 08: `validate_contract` + a single `decide` |
| Big-bang policy rewrites | 08: one narrow change with a decision diff |

Every row is also an attack in the gauntlet above; the **Shortcuts** column is this table, executed.

## Final challenge — Autonomy Review Board (§2.14)
The verdict cell above is a one-line board. Turn it into a function
`review(v1, v2, changes, hypothesis) -> "promote" | "hold" | "rollback"` that:

1. promotes only if every SLO passes under **v2** *and* the full gauntlet still scores 100%;
2. holds if the change touches more than one action or lacks a stated hypothesis;
3. rolls back if any SLO regresses relative to v1.

Then **make the case for v2**. Propose the separate, reviewed change to the risk definition that
would let a *bounded* grant stop counting as an unpaused high-risk action. Why must it be a separate
step, and what new attack in `harnesslab/arena.py` would catch someone abusing it? Expand authority only
where evidence supports it.